# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# どう出来ているか —— 構造・設定・落とし穴・設計の約束

**途中から入る人が最初に読むもの。** 何がどこにあり、どこを踏むと危ないか。
(以前の `handover.md` の内容。経緯の細部は [../Old/docs/handover.md](../Old/docs/handover.md))

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 1. これは何か

**学校の構内地図と、その管理ダッシュボード。**

| 面 | 入口 | 認証 |
|---|---|---|
| **公開ページ** | `https://ito4.jp/` | 不要(一部の機能だけ解除コード) |
| **管理画面** | `https://admin.ito4.jp/admin/` | Logto でサインイン |
| **Android アプリ** | 別リポジトリ | Logto でサインイン(Bearer トークン) |

サーバー側が Android に提供するのは `src/api/*.php` の口だけ。

## 2. リポジトリの構造

```
Website/
  index.html  pages/  css/  js/  …   ← AdminLTE v4.2.0 の原本(**編集しない**。直すなら server/src/admin 側で上書き)
  server/                            ← 本体
    compose.yaml / compose.vps.yaml  基本構成と本番の差分
    .env.example                     秘密の見本(実物の .env はリポジトリに入れない)
    docker/  nginx/                  web / soketi の Dockerfile、nginx 本体と部品(km/)
    scripts/                         運用スクリプト(PowerShell と、ホストに置く .sh)
    docs/                            **この取扱説明書**(.ipynb)と status.md / plan.md
    Old/                             役目を終えたもの(配備されない)。以前の文書は Old/docs
    src/                             PHP のアプリ本体。web コンテナに丸ごと mount される
```

### `src/` の中身

| 置き場 | 中身 |
|---|---|
| `index.php` `contact.php` `faq.php` `terms.php` `privacy.php` | 公開ページ |
| `sign-in.php` `sign-out.php` `callback.php` / `logto_guard.php` | Logto のサインイン往復 / **Android 用 Bearer 検証** |
| `api/` | Android 向けの口。`floor-image.php` は見取り図を錠の内側から配る |
| `admin/` | 管理画面(`_inc/` が土台、`api/` が画面用の口) |
| `lib/` | **ロジックの本体**(下) |
| `Main/` | 公開ページの地図の JS/CSS |
| `config/` | `*.example.php` が見本、`*.local.php` が実物(**配備されない**) |
| `scripts/check.php` | **自己検査は1本** |
| `uploads/` | アップロードされたファイルと**配信する地図 JSON**(www-data の持ち物) |

| `lib/` の読みどころ | 何をするか |
|---|---|
| `site.php` | **URL とポートの正本**(`APP_URL` から組む) |
| `db.php` | MariaDB 接続(TLS 検証あり) |
| `logto-management.php` | Logto Management API。ユーザー一覧・**停止判定** |
| `map-data.php` / `map-edit.php` / `map-events.php` | 地図 / 編集 / イベントモード |
| `mailer.php` | メール送信(すべて環境変数から) |
| `csp.php` / `recaptcha.php` | Content-Security-Policy / 公開フォームの bot よけ |
| `*-notice.php` | 定期の知らせの本文(update / security / backup / log) |

## 3. 動いているもの

```
                     インターネット
        80 443 3001 3002 6001 8281        (8025 は本番では開けない)
                          |
                  [reverse-proxy] nginx
        ┌──────────┬────────┴─┬──────────┬──────────┐
      [web]   [phpmyadmin] [logto]   [soketi]   ([mailpit] はローカル環境だけ)
   php-apache                 |
   [mariadb]             [postgres]
   [certbot] 証明書更新    [mailserver] 送信専用(外に出ていない)
```

- **外に出ているのは `reverse-proxy` だけ。** `3306` と mailserver の `587` は公開しない
- **本番で動くのは 9 つ**(`certbot logto mailserver mariadb phpmyadmin postgres reverse-proxy soketi web`)。`mailpit` は `--profile mailpit` のローカル環境用で、本番では動かさない。**2026-09-18 から `8025` も publish しない**(誰も居ない口を開けない。ローカル環境では `compose.local.yaml` が足す)
- `3001`(Logto)と `6001`(Soketi)はブラウザが直接つなぐので公開が必須
- `3002` / `8281`(ローカル環境では `8025` も)は管理系。**Logto のログイン判定**の後ろ
  (`nginx/km/gate-location.conf` が `auth_request` で `admin/api/gate.php` を呼び、セッションとアカウント停止を見てから流す。**判定は `proxy_pass` の手前**)

## 4. 設定の読み方

値は3段階で解決され、**先に見つかった方が勝つ。**

1. 環境変数(`.env` → compose → コンテナ)
2. `src/config/*.local.php`
3. コード内の既定値

`.env` と `*.local.php` は**配備されない**(ホストが正本)。何を書くかは `.env.example` に理由つきで書いてある。

## 5. 環境

| | この PC | 本番 |
|---|---|---|
| 場所 | `C:\Users\itota\Documents\Website` | `ito4.jp`(`163.43.218.158`)/ `/opt/kosenmap`(持ち主は `kmops`) |
| 中身 | 編集と検査だけ(**Docker は動かさない**) | Ubuntu / 9 サービス(2026-09-18 実測) |

受け皿(検証環境)は無い。**下見(`-WhatIfOnly`)と `check.php` を先に通す。**

## 6. 文字コードの決まり

| 種類 | BOM | 理由 |
|---|---|---|
| `*.ps1` | **付ける** | Windows PowerShell 5.1 は BOM が無いと CP932 として読み、日本語が化けて構文が壊れる |
| `*.sh` | **付けない** | 先頭は `#!/bin/sh` でなければならない |
| `*.php` | **付けない** | 出力がヘッダーより先に出て壊れる |

`check.php` の `powershell` / `shell` の節が見張る。

## 7. 絶対に配備しない・消さないもの

- 配備しない: `.env` / `certs/` / `src/config/*.local.php`
- **本番のボリュームで消してはいけない**: `mail_dkim`(DKIM 秘密鍵。消すと DNS 更新が要る)/ `letsencrypt` / `mariadb_data` / `postgres_data`

## 8. 踏みやすい落とし穴

| 落とし穴 | 対策 |
|---|---|
| `.env` の `COMPOSE_FILE=compose.yaml:compose.vps.yaml` を消す | **消すと `3306` が外に出る** |
| compose の `ports:` / `volumes:` は上書きでなく**併合** | 差し替えたいときは `!override` |
| `.env` の値に `=` を余分に打つ(`KEY=="value"`) | 値が `="value"` になる。**2026-08-28 に管理画面が全面停止した** |
| `.env` の値に `"` `'` `\` `$` | **docker compose が .env 全体を拒む**。メールもバックアップも止まる(2026-09-13) |
| `docker compose exec -T` は後続のスクリプトを飲み込む | パイプで渡すときは `</dev/null` |
| `/etc/cron.d/` のファイルが root 所有でない | **cron が黙って無視する**(2026-09-09) |
| Windows のフォルダの「読み取り専用」 | tar がモード 555 として運ぶ。配備が展開の前後で直す(2026-09-13) |
| この PC のセキュリティソフトが 25/465/587 を横取りする | 存在しない IP でも接続に成功する。**バナーを読んで判断する** |
| コンテナを足す前に `free -h` を見ない | 本番は 1.9Gi。メモリを見ずに増やして落としたことがある |

## 9. 設計上の約束

- **CDN は使わない**(唯一の例外は reCAPTCHA)。QR も外部のサービスを使わない
- **認証は閉じる方に倒す。** ただし補助的な資格情報の不調では閉じない(停止判定が `readonly` で失敗したら `default` で1回だけ試し直す)
- **公開の口は Management API を叩かない**(未ログインの経路はキャッシュを読むだけ)
- **URL は `APP_URL` が正本。** ホスト名を直書きしない
- **秘密はリポジトリに置かない。** 見本だけ置く。**秘密をコマンド行に載せない**(`ps` に出る)
- **CSP により `style="…"` 属性は使わない**
- **当てるのは人。機械は知らせるだけ。** 沈黙を正常と読ませない(`--heartbeat`)
- **文書に理由を書く。** 各ファイルの冒頭コメントに「なぜそうしたか」がある。直す前にそこを読む